# Optimisation du Chiffre d'Affaires — Analyse Retail
## Étape 1 : Exploration des données avec Polars et DuckDB

Dans ce notebook, j'utilise Polars pour le chargement des données (ultra-rapide) et DuckDB pour réaliser des jointures complexes en SQL directement sur les DataFrames. Mon objectif est d'analyser le chiffre d'affaires par produit et d'identifier les catégories les plus performantes.

In [ ]:
import polars as pl
import duckdb
import plotly.express as px

# Chemins vers les fichiers
data_path = "../data/"

print("Chargement des données avec Polars...")
orders = pl.read_csv(data_path + "olist_orders_dataset.csv")
items = pl.read_csv(data_path + "olist_order_items_dataset.csv")
products = pl.read_csv(data_path + "olist_products_dataset.csv")
sellers = pl.read_csv(data_path + "olist_sellers_dataset.csv")

print(f"Commandes : {orders.shape[0]}, Lignes de commande : {items.shape[0]}")

### 1. Jointure des Données (DuckDB)
Je réunis toutes les données utiles en une seule table.

In [ ]:
query = """
SELECT 
    i.order_id,
    o.order_purchase_timestamp,
    i.product_id,
    COALESCE(p.product_category_name, 'inconnue') AS category,
    i.seller_id,
    i.price,
    i.freight_value
FROM items i
JOIN orders o ON i.order_id = o.order_id
LEFT JOIN products p ON i.product_id = p.product_id
JOIN sellers s ON i.seller_id = s.seller_id
WHERE o.order_status = 'delivered'
"""

print("Exécution de la requête SQL via DuckDB...")
df_master = duckdb.query(query).pl()

print("Aperçu des données croisées :")
print(df_master.head())

### 2. Analyse du Chiffre d'Affaires par Catégorie
Je calcule le CA total généré par chaque catégorie.

In [ ]:
category_ca = df_master.group_by('category').agg([
    pl.sum('price').alias('total_ca'),
    pl.len().alias('sales_volume')
])

top_ca = category_ca.sort('total_ca', descending=True).head(10)

print("--- Top 10 Catégories par Chiffre d'Affaires ---")
print(top_ca)


### 3. Distribution du CA par Vendeur (Sellers)
Quels vendeurs génèrent le plus de CA ?

In [ ]:
seller_ca = df_master.group_by('seller_id').agg([
    pl.sum('price').alias('total_ca')
]).sort('total_ca', descending=True)

fig = px.histogram(
    seller_ca.to_pandas(), 
    x='total_ca', 
    nbins=50, 
    title="Distribution du CA Total par Vendeur",
    color_discrete_sequence=['#3498db']
)
fig.show()

## Conclusions Business

1. **Concentration du CA** : Une petite partie des catégories (comme la santé/beauté ou les montres/cadeaux) génère une part disproportionnée du chiffre d'affaires.
2. **Hétérogénéité des vendeurs** : La plupart des vendeurs réalisent un très petit CA, tandis qu'une poignée domine le marché. La plateforme dépend fortement de ces gros vendeurs.